# R16c step 6 · Aligned crops for the grader: does lining up each pair help find-then-grade? (Kaggle T4)

Task **R16c**, step 6. Built 17 Nov 2026, at Abrar's go. It trains nothing: the three graders of R16c steps 5 and 5b grade the same SAM 3 outlines again, with the after-photo aligned to the before-photo.

**Why**
- Ridita's audit found the two photos of a pair don't line up: 52% of 2,966 measurable pairs are shifted by more than 2 px, 10% by more than 5 px, and the worst by 56 px (Task_Ledger §9).
- The grader compares 128 px crops of one building, before and after. A shift moves the building inside the after-crop, so an intact roof can look displaced, like damage.
- With perfect grades, find-then-grade would reach destroyed IoU 0.556 on validation, against our graders' 0.3065 (mean of three). Most of the loss is in grading.

**What it does, on the 605 clean validation pairs**
- **V0, as before:** crops exactly as R16c step 5 made them. This is the check: it must reproduce the saved grades and scores.
- **V1, pair-aligned:** one shift per pair is measured by phase correlation between the whole before-photo and after-photo. The after-photo is moved back by it before cropping.
  - It's the after-photo that moves, because SAM 3's outlines were drawn on the before-photo, so the outlines stay put.
  - The shift is used only if the match is clear (response at least 0.05) and at most 64 px.
- **V2, building-aligned:** V1, then a second, local shift per outline, from phase correlation on a 192 px window around it. It is at most 16 px, and is used only if its match is clear.
- **The same three graders (seeds 42, 43, 44),** with their saved label biases, grade every outline under each version. The map is the outlines filled with their grades, and is scored against the truth exactly as before. Only the grades change.

**Checks (the notebook stops if one fails)**
1. V0's log-probabilities match the saved ones within 0.05 on at least 99.5% of outlines. E7's limit was 0.05.
2. V0's destroyed IoU per seed reproduces R16c's: 0.3161, 0.3051 and 0.2982, within 0.0005. For seed 43 the limit is 0.0015, because its saved file rebuilds only to within 0.0012 (R16c step 5; E7 step 2c).
3. On a made-up image, phase correlation recovers a known shift to within 0.25 px (Cell 4).

**The rule (fixed before the run)**
- Of V1 and V2, the one with the higher destroyed IoU (mean of the three seeds) on the 605 clean pairs is the candidate.
- It passes if its gain over V0 is more than noise: the pairs are redrawn 10,000 times, as in E7 step 2b, and the middle 95% of the gain must lie above 0.
- **If it passes:** a step 6b confirms it once on the 2,007 test pairs, and retraining the grader on aligned crops becomes the next candidate.
- **If not:** alignment at grading time doesn't help, and that is recorded.

**Reported beside it, deciding nothing**
- mIoU and building IoU;
- the scores on xBD pairs and on the other sources separately. Option (c)'s pick (E7 step 2c) uses find-then-grade only on the other sources, Turkey and the rest.
- The grades against the truth under each outline: the outline's main true class, background read as "not a building";
- The measured shifts: the share above 2 px and above 5 px, and the largest, beside Ridita's 52%, 10% and 56 px.

**Expectation (fixed before the run; rough guesses)**
- V0 reproduces the saved scores.
- About half the pairs are shifted by more than 2 px, as Ridita found.
- The gain is small, +0.00 to +0.02 destroyed IoU, mostly on the 10% of pairs shifted by more than 5 px. The graders were trained on unaligned crops, so they may already tolerate small shifts, and aligning only at grading time may not change much. The real gain may need a grader retrained on aligned crops.

**How to run on Kaggle** (account 1, where the inputs are)
1. Settings: GPU **T4 x2** (one is used), internet **on**. No secret is needed.
2. Inputs:
   - `abrarmohammedtanzim/disasterm3-mirror`;
   - Your Work → `e3-sam3-building-finding`;
   - Your Work → `r16c-step5-grader-on-sam3-outlines`;
   - Your Work → `r16c-step5b-third-grader-seed`.
3. Run All once with `SMOKE_TEST = True` in Cell 2 (20 clean pairs; a few minutes). Then set it to `False`, and use Save & Run All. About 20–30 minutes, estimated.

**Outputs** in `/kaggle/working/step6/`:
- `README.md`, `config.json`;
- `scores.csv`, `resampling.csv`, `shifts.csv`;
- `grades_aligned.csv`: every outline's grade per version and seed, for step 6b and option (c).

In [1]:
# ── Cell 1: Environment (Kaggle's own packages; nothing to install) ──────────
import os
import sys

import cv2
import numpy as np
import torch
import torchvision

print(f"✓ torch {torch.__version__}, torchvision {torchvision.__version__}, OpenCV {cv2.__version__}, "
      f"python {sys.version.split()[0]}")
print(f"  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (a GPU is needed for speed)'}")

✓ torch 2.11.0+cu130, torchvision 0.26.0+cu130, OpenCV 4.14.0, python 3.13.16
  GPU: Tesla T4


In [2]:
# ── Cell 2: Configuration and inputs ──────────────────────────────────────
import json
import random
import time
from collections import defaultdict
from pathlib import Path

import pandas as pd

SMOKE_TEST = False                 # True = the first 20 clean pairs
INPUT_ROOT = Path("/kaggle/input")
DATA_ROOT = INPUT_ROOT / "datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct"
MANIFEST_PATH = DATA_ROOT / "train_release.json"
OUT_DIR = Path("/kaggle/working/step6")
OUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 4                    # 0 background, 1 intact, 2 damaged, 3 destroyed
LABELS = ["Intact", "Damaged", "Destroyed", "None"]       # the graders' labels
SEEDS = (42, 43, 44)
BIAS_FIXED = {42: [0.0, -1.75, -4.0, -1.25], 43: [0.0, -1.0, -4.0, -2.0], 44: [0.0, -2.0, -4.0, -3.5]}
CROP, MARGIN = 128, 32             # as the graders trained (R16c steps 2 and 5)
MAX_SHIFT, LOCAL_MAX = 64, 16      # px: the largest pair shift and local shift used (the audit's worst was 56)
R_MIN = 0.05                       # phase correlation's response below this: no clear match, no shift
LOCAL_WIN = 192                    # px: the window around an outline for its local shift
PUBLISHED = {42: 0.3161, 43: 0.3051, 44: 0.2982}          # R16c steps 5 and 5b, 605 clean pairs
SCORE_TOL = {42: 0.0005, 43: 0.0015, 44: 0.0015}          # seed 43's file rebuilds only to within 0.0012
LOGP_TOL, LOGP_SHARE = 0.05, 0.995
N_RESAMPLES, RNG_SEED = 10_000, 0
# E5 step 1's 40 validation pairs that share images with training (val_train_leaks_hash.csv, counted True)
LEAKED = {f"socal_fire_{n:08d}_post_disaster.png" for n in
          (81, 188, 218, 220, 222, 235, 301, 341, 348, 396, 436, 455, 497, 504, 614, 1188)} | \
         {f"woolsey_fire_{n:08d}_post_disaster.png" for n in
          (9, 30, 32, 137, 272, 399, 435, 474, 498, 514, 529, 573, 576, 589, 623, 632, 634, 653, 679, 703, 722, 758,
           783, 810)}
assert len(LEAKED) == 40

FOUND, FOUND_DIRS = defaultdict(list), defaultdict(list)
for root, dirs, files in os.walk(INPUT_ROOT):
    for name in files:
        if name in ("instances_val.csv", "calibration.json") or name.startswith("grader_seed"):
            FOUND[name].append(Path(root) / name)
    for name in dirs:
        if name == "pre_grid2":
            FOUND_DIRS[name].append(Path(root) / name)
WHY = "attach Your Work → r16c-step5-grader-on-sam3-outlines and r16c-step5b-third-grader-seed"
GRADER_PATHS = {}
for s in SEEDS:
    hits = sorted(FOUND[f"grader_seed{s}.pth"])
    if not hits:
        raise FileNotFoundError(f"No grader_seed{s}.pth: {WHY}")
    GRADER_PATHS[s] = hits[0]
BIASES = {}
for path in FOUND["calibration.json"]:
    for seed, chosen in json.loads(path.read_text(encoding="utf-8")).items():
        if int(seed) in SEEDS and [chosen["bias"][l] for l in LABELS] == BIAS_FIXED[int(seed)]:
            BIASES[int(seed)] = np.array(BIAS_FIXED[int(seed)])
assert sorted(BIASES) == list(SEEDS), f"Biases found for {sorted(BIASES)} only: {WHY}"
INST_PATHS = {}
for path in FOUND["instances_val.csv"]:
    cols = set(pd.read_csv(path, nrows=0).columns)
    for s in SEEDS:
        if {f"logp_{l.lower()}_s{s}" for l in LABELS} <= cols:
            INST_PATHS[s] = path
assert sorted(INST_PATHS) == list(SEEDS), f"Saved grades found for {sorted(INST_PATHS)} only: {WHY}"
if not FOUND_DIRS["pre_grid2"]:
    raise FileNotFoundError("No pre_grid2 folder (SAM 3's saved maps): attach Your Work → e3-sam3-building-finding")
E3_DIR = sorted(FOUND_DIRS["pre_grid2"])[0]
assert MANIFEST_PATH.exists(), "attach abrarmohammedtanzim/disasterm3-mirror"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ R16c step 6{'  [SMOKE TEST]' if SMOKE_TEST else ''}: graders {GRADER_PATHS}")
print(f"  SAM 3 maps: {E3_DIR}; saved grades: {INST_PATHS}")

✓ R16c step 6: graders {42: PosixPath('/kaggle/input/notebooks/abrarmohammedtanzim/r16c-step5-grader-on-sam3-outlines/step5/grader_seed42.pth'), 43: PosixPath('/kaggle/input/notebooks/abrarmohammedtanzim/r16c-step5-grader-on-sam3-outlines/step5/grader_seed43.pth'), 44: PosixPath('/kaggle/input/notebooks/abrarmohammedtanzim/r16c-step5b-third-grader-seed/step5b/grader_seed44.pth')}
  SAM 3 maps: /kaggle/input/notebooks/abrarmohammedtanzim/e3-sam3-building-finding/E3/masks/pre_grid2; saved grades: {44: PosixPath('/kaggle/input/notebooks/abrarmohammedtanzim/r16c-step5b-third-grader-seed/step5b/instances_val.csv'), 42: PosixPath('/kaggle/input/notebooks/abrarmohammedtanzim/r16c-step5-grader-on-sam3-outlines/step5/instances_val.csv'), 43: PosixPath('/kaggle/input/notebooks/abrarmohammedtanzim/r16c-step5-grader-on-sam3-outlines/step5/instances_val.csv')}


In [3]:
# ── Cell 3: The validation pairs (the D4 split, as in L1c) and the 605 clean ones ──
from sklearn.model_selection import train_test_split

with open(MANIFEST_PATH, encoding="utf-8") as f:
    raw = json.load(f)


def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    for c in (DATA_ROOT / rel_path, DATA_ROOT / "train_images" / rel_path, DATA_ROOT / "train_images" / "train_images" / filename,
              DATA_ROOT / "masks" / rel_path, DATA_ROOT / "masks" / "masks" / filename):
        if c.exists():
            return str(c)
    return None


by_image = defaultdict(dict)       # the D4 notebooks' Cell 4, with the before-photo (R11's version)
for e in raw:
    if e.get("task") != "Referring Expression Segmentation" or e.get("cls_description") != "Building Damage Assessment":
        continue
    if e.get("image_type") != "Optical":
        continue
    post_img = e.get("post_image_path", "")
    folder = Path(e.get("ground_truth", "").replace("\\", "/")).parent.name
    resolved = resolve_path(e.get("ground_truth", ""))
    if resolved:
        by_image[post_img][folder] = resolved
        by_image[post_img]["pre_image_path"] = e.get("pre_image_path")
del raw
FOLDER_TO_CLASS = {"train_building_intact_mask": 1, "train_building_damaged_mask": 2, "train_building_destroyed_mask": 3}
pairs = []
for post_img, d in by_image.items():
    img_path = resolve_path(post_img)
    if not img_path or not any(k in d for k in FOLDER_TO_CLASS):
        continue
    pairs.append({"image_path": img_path, "pre_image_path": resolve_path(d.get("pre_image_path")),
                  "masks": {k: d[k] for k in FOLDER_TO_CLASS if k in d}})
_, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)
clean = [p for p in val_pairs if Path(p["image_path"]).name not in LEAKED]
assert len(val_pairs) == 645 and len(clean) == 605, f"{len(val_pairs)} validation pairs, {len(clean)} clean"
assert all(p["pre_image_path"] for p in clean), "a clean pair has no before-photo"
if SMOKE_TEST:
    clean = clean[:20]


def truth_of(pair, shape):
    """The D4 rule: intact, then damaged, then destroyed, each written over the last; at the photo's size."""
    truth = None
    for folder, cls in FOLDER_TO_CLASS.items():
        if folder in pair["masks"]:
            m = cv2.imread(pair["masks"][folder], cv2.IMREAD_GRAYSCALE)
            if m is None:
                continue
            truth = np.zeros_like(m, np.uint8) if truth is None else truth
            truth[m > 0] = cls
    if truth.shape != shape:
        truth = cv2.resize(truth, (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST)
    return truth


print(f"✓ {len(pairs):,} pairs; validation {len(val_pairs)}; clean {len(clean)}{' (SMOKE TEST: first 20)' if SMOKE_TEST else ''}")

✓ 6,443 pairs; validation 645; clean 605


In [4]:
# ── Cell 4: Helpers: E7 step 2's grader code (unchanged), the alignment, the scoring; with self-checks ──
import torch.nn as nn
from scipy import ndimage


def square_box(sl, h, w):
    """Square window around a building's (row, column) slices, MARGIN px wider on each side, kept inside the image."""
    y0, y1, x0, x1 = sl[0].start, sl[0].stop, sl[1].start, sl[1].stop
    side = min(max(y1 - y0, x1 - x0) + 2 * MARGIN, h, w)
    cy, cx = (y0 + y1) / 2, (x0 + x1) / 2
    top = int(round(min(max(cy - side / 2, 0), h - side)))
    left = int(round(min(max(cx - side / 2, 0), w - side)))
    return top, left, side


def building_crops(pre, post, labels, ids, offsets=None):
    """uint8 (len(ids), 7, CROP, CROP): before RGB, after RGB, and 255 on the building. R16c step 5's code, plus an
    optional whole-pixel offset (dx, dy) per outline for the after-window (V2): the after-photo is read from a
    reflect-padded copy, so a window shifted past the edge still has content."""
    h, w = labels.shape
    slices = ndimage.find_objects(labels)
    pad = LOCAL_MAX
    post_p = cv2.copyMakeBorder(post, pad, pad, pad, pad, cv2.BORDER_REFLECT) if offsets is not None else None
    out = np.empty((len(ids), 7, CROP, CROP), np.uint8)
    for n, lid in enumerate(ids):
        top, left, side = square_box(slices[lid - 1], h, w)
        win = (slice(top, top + side), slice(left, left + side))
        out[n, 0:3] = cv2.resize(pre[win], (CROP, CROP), interpolation=cv2.INTER_LINEAR).transpose(2, 0, 1)
        if offsets is None:
            after = post[win]
        else:
            dx, dy = offsets[n]
            after = post_p[top + pad + dy: top + pad + dy + side, left + pad + dx: left + pad + dx + side]
        out[n, 3:6] = cv2.resize(after, (CROP, CROP), interpolation=cv2.INTER_LINEAR).transpose(2, 0, 1)
        out[n, 6] = cv2.resize((labels[win] == lid).astype(np.uint8) * 255, (CROP, CROP), interpolation=cv2.INTER_NEAREST)
    return out


class SiameseGrader(nn.Module):
    """A shared ResNet-18 reads the before and the after crop (RGB + building mask each);
    [f_before, f_after, f_after - f_before] -> one score per label. R16c step 5's model."""

    def __init__(self):
        super().__init__()
        net = torchvision.models.resnet18(weights=None)
        net.conv1 = nn.Conv2d(4, 64, kernel_size=7, stride=2, padding=3, bias=False)
        net.fc = nn.Identity()
        self.encoder = net
        self.head = nn.Sequential(nn.Linear(3 * 512, 256), nn.ReLU(inplace=True), nn.Dropout(0.2), nn.Linear(256, len(LABELS)))

    def forward(self, before, after):
        fb, fa = self.encoder(before), self.encoder(after)
        return self.head(torch.cat([fb, fa, fa - fb], 1))


MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1) * 255
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1) * 255


def to_inputs(x):
    """uint8 (B, 7, S, S) -> normalised before and after tensors (B, 4, S, S); the mask becomes -1 / +1."""
    x = x.to(DEVICE, non_blocking=True).float()
    mask = x[:, 6:7] / 127.5 - 1
    return torch.cat([(x[:, 0:3] - MEAN) / STD, mask], 1), torch.cat([(x[:, 3:6] - MEAN) / STD, mask], 1)


@torch.no_grad()
def log_probs(model, crops):
    """uint8 crops (N, 7, S, S) -> log-probabilities (N, labels)."""
    model.eval()
    out = []
    for i in range(0, len(crops), 512):
        before, after = to_inputs(torch.from_numpy(np.ascontiguousarray(crops[i:i + 512])))
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            out.append(torch.log_softmax(model(before, after).float(), 1).cpu())
    return torch.cat(out).numpy() if out else np.zeros((0, len(LABELS)), np.float32)


def load_grader(path):
    model = SiameseGrader()
    model.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))
    return model.to(DEVICE).eval()


def to_class(logp, seed):
    """Grades with the seed's biases: intact, damaged, destroyed -> 1-3; "None" (not a building) -> 0."""
    label = (logp + BIASES[seed]).argmax(1)
    return np.where(label < 3, label + 1, 0)


# ── The alignment ──
def gray(rgb):
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)


def measure_shift(a, b, limit):
    """The shift of b relative to a (phase correlation, Hanning window), as (dx, dy, response, used). It is used only if
    the match is clear (response >= R_MIN) and within the limit; otherwise (0, 0)."""
    win = cv2.createHanningWindow((a.shape[1], a.shape[0]), cv2.CV_32F)
    (dx, dy), response = cv2.phaseCorrelate(a, b, win)
    used = response >= R_MIN and max(abs(dx), abs(dy)) <= limit
    return (dx, dy, response, used) if used else (0.0, 0.0, response, False)


def move(img, dx, dy):
    """Moves the picture's content by (dx, dy); the edges are filled by reflection."""
    return cv2.warpAffine(img, np.float32([[1, 0, dx], [0, 1, dy]]), (img.shape[1], img.shape[0]), borderMode=cv2.BORDER_REFLECT)


def local_offsets(pre_g, post_g, labels, ids):
    """V2: per outline, the shift of the after-photo relative to the before-photo in a LOCAL_WIN window around it."""
    h, w = labels.shape
    slices = ndimage.find_objects(labels)
    half = LOCAL_WIN // 2
    out, used = [], 0
    for lid in ids:
        sl = slices[lid - 1]
        cy, cx = (sl[0].start + sl[0].stop) // 2, (sl[1].start + sl[1].stop) // 2
        y0, x0 = min(max(cy - half, 0), h - LOCAL_WIN), min(max(cx - half, 0), w - LOCAL_WIN)
        dx, dy, _, ok = measure_shift(pre_g[y0:y0 + LOCAL_WIN, x0:x0 + LOCAL_WIN], post_g[y0:y0 + LOCAL_WIN, x0:x0 + LOCAL_WIN], LOCAL_MAX)
        out.append((int(round(dx)), int(round(dy))))
        used += ok
    return out, used


# ── The scoring (as E7 and R16c: the outlines filled with their grades, against the truth) ──
def confusion(truth, pred):
    return np.bincount(truth.ravel().astype(np.int64) * NUM_CLASSES + pred.ravel(), minlength=NUM_CLASSES ** 2).reshape(4, 4)


def scores(cm):
    tp = np.diag(cm).astype(np.float64)
    iou = tp / np.maximum(cm.sum(0) + cm.sum(1) - tp, 1)
    b = cm[1:, 1:].sum()
    return {"miou": iou.mean(), "destroyed_iou": iou[3], "building_iou": b / max(b + cm[0, 1:].sum() + cm[1:, 0].sum(), 1)}


# Self-checks: a known shift is recovered and undone; the crops' shape; the grades with biases; the scoring
_rng = np.random.default_rng(0)
_a = cv2.GaussianBlur(_rng.random((256, 256)).astype(np.float32), (0, 0), 3)
_b = move(_a, 5, -3)
_dx, _dy, _r, _ok = measure_shift(_a, _b, MAX_SHIFT)
assert _ok and abs(_dx - 5) < 0.25 and abs(_dy + 3) < 0.25, (_dx, _dy)
assert np.abs(move(_b, -_dx, -_dy) - _a)[20:-20, 20:-20].mean() < 0.03
_lab = np.zeros((200, 300), np.int32)
_lab[0:10, 0:20], _lab[100:150, 200:260] = 1, 2
_img = np.zeros((200, 300, 3), np.uint8)
assert building_crops(_img, _img, _lab, [1, 2]).shape == (2, 7, CROP, CROP)
assert building_crops(_img, _img, _lab, [1, 2], offsets=[(3, -2), (0, 0)]).shape == (2, 7, CROP, CROP)
assert to_class(np.array([[-0.1, -9, -9, -9], [-9, -9, -9, -0.1]]), 42).tolist() == [1, 0]
_t = np.array([[0, 3], [3, 1]], np.uint8)
assert abs(scores(confusion(_t, _t))["destroyed_iou"] - 1) < 1e-9
print("✓ Helpers ready (self-checks passed: a 5, -3 px shift recovered and undone)")

✓ Helpers ready (self-checks passed: a 5, -3 px shift recovered and undone)


In [5]:
# ── Cell 5: Every clean pair: the shift, the crops of V0, V1 and V2, and the three graders' grades ──
from tqdm.auto import tqdm

graders = {s: load_grader(GRADER_PATHS[s]) for s in SEEDS}
saved = {s: pd.read_csv(INST_PATHS[s]).query("found_by == 'sam3'") for s in SEEDS}
VARIANTS = ("V0", "V1", "V2")
LOGP_COLS = {s: [f"logp_{l.lower()}_s{s}" for l in LABELS] for s in SEEDS}

records, shifts = [], []
CMS = {(v, s): {} for v in VARIANTS for s in SEEDS}       # per image confusion, per version and seed
logp_check = []                                           # V0 against the saved log-probabilities
t0 = time.time()
for p in tqdm(clean, desc="pairs"):
    name = Path(p["image_path"]).name
    post = cv2.cvtColor(cv2.imread(p["image_path"]), cv2.COLOR_BGR2RGB)
    pre = cv2.cvtColor(cv2.imread(p["pre_image_path"]), cv2.COLOR_BGR2RGB)
    if pre.shape != post.shape:                           # as E7 and the D4 dataset class
        pre = cv2.resize(pre, (post.shape[1], post.shape[0]), interpolation=cv2.INTER_LINEAR)
    truth = truth_of(p, post.shape[:2])
    sam = cv2.imread(str(E3_DIR / f"{Path(name).stem}.png"), cv2.IMREAD_UNCHANGED)
    sam = np.zeros(post.shape[:2], np.int32) if sam is None else sam.astype(np.int32)
    if sam.shape != post.shape[:2]:
        sam = cv2.resize(sam, (post.shape[1], post.shape[0]), interpolation=cv2.INTER_NEAREST)
    rows = saved[42][saved[42]["image"] == name]
    ids = rows["id"].tolist()

    # V1: one shift per pair, the after-photo moved back onto the before-photo
    dx, dy, resp, used = measure_shift(gray(pre), gray(post), MAX_SHIFT)
    post_v1 = move(post, -dx, -dy) if used else post
    shifts.append({"image": name, "dx": round(dx, 2), "dy": round(dy, 2), "response": round(float(resp), 4), "used": used})

    if ids:
        crops = {"V0": building_crops(pre, post, sam, ids), "V1": building_crops(pre, post_v1, sam, ids)}
        offsets, n_local = local_offsets(gray(pre), gray(post_v1), sam, ids)
        crops["V2"] = building_crops(pre, post_v1, sam, ids, offsets=offsets)
        under = np.bincount(sam.ravel().astype(np.int64) * NUM_CLASSES + truth.ravel(),
                            minlength=(int(sam.max()) + 1) * NUM_CLASSES).reshape(-1, NUM_CLASSES)[np.array(ids)]
        main_truth = under.argmax(1)                      # the outline's main true class (0 = mostly background)
    for s in SEEDS:
        for v in VARIANTS:
            classes = np.zeros(0, np.int64)
            if ids:
                lp = log_probs(graders[s], crops[v])
                classes = to_class(lp, s)
                if v == "V0":
                    mine = saved[s][saved[s]["image"] == name].set_index("id").loc[ids, LOGP_COLS[s]].to_numpy()
                    logp_check.append(np.abs(lp - mine).max(1))
            lut = np.zeros(int(sam.max()) + 1, np.uint8)
            if ids:
                lut[np.array(ids)] = classes
            CMS[(v, s)][name] = confusion(truth, lut[sam])
            if ids:
                records.extend({"image": name, "id": i, "variant": v, "seed": s, "grade": int(c), "main_truth": int(t)}
                               for i, c, t in zip(ids, classes, main_truth))
    if ids:
        shifts[-1]["local_shifts_used"] = n_local
        shifts[-1]["outlines"] = len(ids)

GRADES = pd.DataFrame(records)
SHIFTS = pd.DataFrame(shifts)
LOGP_DIFF = np.concatenate(logp_check) if logp_check else np.zeros(0)
print(f"✓ {len(clean)} pairs, {GRADES['id'].nunique() if len(GRADES) else 0} outline ids, in {(time.time() - t0) / 60:.1f} min")
print(f"  Pair shifts used: {int(SHIFTS['used'].sum())} of {len(SHIFTS)}; "
      f"|shift| > 2 px: {(SHIFTS[['dx', 'dy']].abs().max(1) > 2).mean():.0%}, > 5 px: {(SHIFTS[['dx', 'dy']].abs().max(1) > 5).mean():.0%}, "
      f"largest {SHIFTS[['dx', 'dy']].abs().max(1).max():.1f} px")

pairs:   0%|          | 0/605 [00:00<?, ?it/s]

✓ 605 pairs, 290 outline ids, in 9.2 min
  Pair shifts used: 483 of 605; |shift| > 2 px: 47%, > 5 px: 17%, largest 48.8 px


In [6]:
# ── Cell 6: Checks, scores, the rule, resampling, and the files ─────────────
import re

IMAGES = [Path(p["image_path"]).name for p in clean]
XBD = re.compile(r"^(.+?)_(\d{8})_post_disaster")
GROUPS = {"all": IMAGES, "xBD": [i for i in IMAGES if XBD.match(Path(i).stem)],
          "other sources": [i for i in IMAGES if not XBD.match(Path(i).stem)]}


def total(cms, images):
    return sum((cms[i] for i in images), np.zeros((4, 4), np.int64))


# Check 1: V0's log-probabilities against the saved ones
share_ok = float((LOGP_DIFF <= LOGP_TOL).mean()) if len(LOGP_DIFF) else 1.0
CHECKS = [{"check": f"V0 log-probabilities within {LOGP_TOL} of the saved ones", "result": f"{share_ok:.4f} of outlines",
           "pass": share_ok >= LOGP_SHARE}]
# Check 2: V0's destroyed IoU per seed against R16c's published scores (only meaningful on all 605 pairs)
for s in SEEDS:
    d = scores(total(CMS[("V0", s)], IMAGES))["destroyed_iou"]
    CHECKS.append({"check": f"V0 destroyed IoU, seed {s}, against {PUBLISHED[s]} (within {SCORE_TOL[s]})", "result": round(float(d), 4),
                   "pass": SMOKE_TEST or abs(d - PUBLISHED[s]) <= SCORE_TOL[s]})
CHECKS = pd.DataFrame(CHECKS)
print(CHECKS.to_string(index=False))
if not CHECKS["pass"].all():
    raise RuntimeError("A check failed: V0 doesn't reproduce the saved grades or scores, so V1 and V2 can't be compared. Stop.")

rows = []
for v in VARIANTS:
    for group, images in GROUPS.items():
        per_seed = [scores(total(CMS[(v, s)], images)) for s in SEEDS]
        g = GRADES[GRADES["variant"] == v] if len(GRADES) else GRADES
        if group != "all" and len(g):
            g = g[g["image"].isin(images)]
        agree = float((g["grade"] == g["main_truth"]).mean()) if len(g) else float("nan")
        rows.append({"variant": v, "pairs": group, "n_pairs": len(images),
                     **{k: round(float(np.mean([x[k] for x in per_seed])), 4) for k in per_seed[0]},
                     **{f"destroyed_iou_s{s}": round(float(x["destroyed_iou"]), 4) for s, x in zip(SEEDS, per_seed)},
                     "grade_agrees_with_main_truth": round(agree, 4)})
SCORES = pd.DataFrame(rows)
print(SCORES.to_string(index=False))

# The rule: of V1 and V2, the higher destroyed IoU (mean of three seeds, all pairs); it passes if its gain over V0 is
# more than noise (the pairs redrawn 10,000 times; the middle 95% above 0)
alls = SCORES[SCORES["pairs"] == "all"].set_index("variant")
CANDIDATE = "V1" if alls.loc["V1", "destroyed_iou"] >= alls.loc["V2", "destroyed_iou"] else "V2"


def parts(v):
    out = []
    for s in SEEDS:
        tp = np.array([CMS[(v, s)][i][3, 3] for i in IMAGES], np.float64)
        un = np.array([CMS[(v, s)][i][3, :].sum() + CMS[(v, s)][i][:, 3].sum() - CMS[(v, s)][i][3, 3] for i in IMAGES], np.float64)
        out.append((tp, un))
    return out


rng = np.random.default_rng(RNG_SEED)
counts = np.stack([np.bincount(r, minlength=len(IMAGES)) for r in rng.integers(0, len(IMAGES), size=(N_RESAMPLES, len(IMAGES)))])
draw = lambda v: np.mean([(counts @ tp) / np.maximum(counts @ un, 1) for tp, un in parts(v)], axis=0)
gain = draw(CANDIDATE) - draw("V0")
RES = pd.DataFrame([{"comparison": f"{CANDIDATE} minus V0", "observed": round(float(alls.loc[CANDIDATE, "destroyed_iou"] - alls.loc["V0", "destroyed_iou"]), 4),
                     "low_2.5%": round(float(np.percentile(gain, 2.5)), 4), "high_97.5%": round(float(np.percentile(gain, 97.5)), 4),
                     "share_above_0": round(float((gain > 0).mean()), 4)}])
PASSES = bool(RES.loc[0, "low_2.5%"] > 0)
print(RES.to_string(index=False))
verdict = (f"{CANDIDATE} passes: its gain over V0 is more than noise. Next: step 6b, once on the test pairs, and retraining the "
           "grader on aligned crops as a candidate." if PASSES else
           f"{CANDIDATE} doesn't pass: its gain over V0 isn't clearly above noise. Aligning at grading time doesn't help these graders.")
print(f"\n{'✓' if PASSES else '✗'} {verdict}")

# Files
SCORES.to_csv(OUT_DIR / "scores.csv", index=False)
RES.to_csv(OUT_DIR / "resampling.csv", index=False)
SHIFTS.to_csv(OUT_DIR / "shifts.csv", index=False)
GRADES.to_csv(OUT_DIR / "grades_aligned.csv", index=False)
sh = SHIFTS[["dx", "dy"]].abs().max(1)
config = {"task": "R16c step 6", "smoke_test": SMOKE_TEST, "pairs": len(IMAGES), "seeds": SEEDS, "biases": {s: BIASES[s].tolist() for s in SEEDS},
          "crop": CROP, "margin": MARGIN, "max_shift": MAX_SHIFT, "local_max": LOCAL_MAX, "local_window": LOCAL_WIN, "r_min": R_MIN,
          "published": PUBLISHED, "score_tolerance": SCORE_TOL, "resamples": N_RESAMPLES, "candidate": CANDIDATE, "passes": PASSES,
          "torch": torch.__version__, "opencv": cv2.__version__}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1, default=str), encoding="utf-8")


def md(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return "```\n" + df.to_string(index=False) + "\n```"


(OUT_DIR / "README.md").write_text("\n".join([
    f"# R16c step 6: aligned crops for the grader{' (SMOKE TEST)' if SMOKE_TEST else ''}", "",
    f"{len(IMAGES)} clean validation pairs; the three graders of R16c steps 5 and 5b; the SAM 3 outlines of E3. V0 = crops as before; "
    "V1 = the after-photo moved back by one phase-correlation shift per pair; V2 = V1 plus a local shift per outline. "
    "The question, the rule and the expectation are in the notebook's first cell.", "",
    "## Checks", "", md(CHECKS), "",
    "## The measured shifts", "",
    f"Pair shifts used: {int(SHIFTS['used'].sum())} of {len(SHIFTS)} (response at least {R_MIN}, at most {MAX_SHIFT} px). "
    f"Shift above 2 px: {(sh > 2).mean():.0%} of pairs; above 5 px: {(sh > 5).mean():.0%}; largest {sh.max():.1f} px "
    "(Ridita's audit: 52%, 10%, 56 px).", "",
    "## Scores (mean of three seeds)", "", md(SCORES), "",
    "## The rule", "", md(RES), "", f"**Verdict:** {verdict}", "",
    "Files: `scores.csv`, `resampling.csv`, `shifts.csv`, `grades_aligned.csv` (every outline's grade per version and seed), `config.json`.",
]) + "\n", encoding="utf-8")
print(f"✓ Wrote README.md, scores.csv, resampling.csv, shifts.csv, grades_aligned.csv and config.json to {OUT_DIR}")

                                                    check             result  pass
       V0 log-probabilities within 0.05 of the saved ones 1.0000 of outlines  True
V0 destroyed IoU, seed 42, against 0.3161 (within 0.0005)             0.3161  True
V0 destroyed IoU, seed 43, against 0.3051 (within 0.0015)             0.3062  True
V0 destroyed IoU, seed 44, against 0.2982 (within 0.0015)             0.2974  True
variant         pairs  n_pairs   miou  destroyed_iou  building_iou  destroyed_iou_s42  destroyed_iou_s43  destroyed_iou_s44  grade_agrees_with_main_truth
     V0           all      605 0.5473         0.3066        0.6307             0.3161             0.3062             0.2974                        0.8182
     V0           xBD      520 0.5758         0.3542        0.6421             0.3643             0.3561             0.3422                        0.8133
     V0 other sources       85 0.4439         0.2623        0.6178             0.2687             0.2616             0.2567